# WINNIIO — Tokyo MRO Digital Twin: Sionna RT v3

Physics-based 3D ray-tracing RF propagation for Altiostar/Rakuten Symphony.

**Pipeline:** OSM buildings → PLY mesh (concrete + glass) → Mitsuba XML (ITU radio materials) → Sionna RT → Coverage maps

**v3 changes (from v2):**
- Wider bbox covering all 22 towers (Sumida Skytree was outside)
- Spatial building selection (prioritize near towers, not just tallest)
- Glass material (ITU) on commercial/tall buildings — RF-transparent facades
- Per-sector power at full rated power (antenna pattern provides isolation)
- Custom antenna pattern (3GPP TR 38.901, 65° HPBW, 18 dBi, 30 dB F/B)
- Diffraction enabled on all bands
- 10x more ray samples (10^5 → 10^6) for smoother coverage
- 4000 buildings (up from 3000)
- UTM zone 54N projection (replaces crude lat/lng formula)
- SRTM 30m DEM terrain — buildings and towers at real elevation
- PLY cache — skip Overpass + mesh rebuild on reruns
- PLATEAU CityGML LOD2 download + parse (with OSM fallback)
- UE mobility simulation — 5 routes, HO detection (success/failure/pingpong)
- Calibration stub (MDT import + RMSE)
- RL hook skeleton (MRO reward function)

**Verified against:** Sionna v2.0.1 API docs, `sionna-large-radio-maps` repo, official tutorials.

---

## 1. Install + Verify GPU

In [ ]:
!pip install -q sionna geojson trimesh shapely pyproj srtm

try:
    import numpy as _np
    _ = _np._core._multiarray_umath._blas_supports_fpe
    print('NumPy OK — no restart needed')
except (AttributeError, ImportError):
    print('NumPy version mismatch — restarting runtime...')
    print('After restart, click Runtime > Run All (passes on second run).')
    import os; os.kill(os.getpid(), 9)

import torch
assert torch.cuda.is_available(), 'No GPU — go to Runtime > Change runtime type > T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Validate Sionna with Built-in Scene

Run Munich first to confirm the API works before building a custom Tokyo scene.

In [ ]:
import sionna.rt
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, RadioMapSolver
import numpy as np

print(f'Sionna version: {sionna.__version__}')

# Quick validation: load built-in Munich scene
test_scene = load_scene(sionna.rt.scene.munich)
test_scene.frequency = 3.7e9

test_scene.tx_array = PlanarArray(
    num_rows=4, num_cols=2,
    vertical_spacing=0.7, horizontal_spacing=0.5,
    pattern='tr38901', polarization='VH'
)
test_scene.rx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='dipole', polarization='cross'
)

test_scene.add(Transmitter(name='test_tx', position=[8.5, 21, 30]))

rm_solver = RadioMapSolver()
test_rm = rm_solver(
    scene=test_scene,
    max_depth=3,
    cell_size=[2, 2],
    samples_per_tx=10**5
)

test_rm.show(metric='path_gain')
print('Sionna RT works. Proceeding to Tokyo.')

## 3. Tower Network Definition

22 towers from the WINNIIO Cesium demo. Exact coordinates, bands, power, sector azimuths.

In [ ]:
towers = [
    {'id':'SJK-001','name':'Shinjuku Station West','lat':35.6896,'lng':139.6982,'height':45,'band':'n77','freq_ghz':3.7,'power_dbm':40,'sectors':3,'azimuth':[0,120,240],'tilt':4,'status':'active'},
    {'id':'SJK-002','name':'Nishi-Shinjuku Tower','lat':35.6935,'lng':139.6917,'height':120,'band':'n77','freq_ghz':3.7,'power_dbm':43,'sectors':3,'azimuth':[30,150,270],'tilt':6,'status':'active'},
    {'id':'SJK-003','name':'Shinjuku Gyoen South','lat':35.6852,'lng':139.7100,'height':35,'band':'n78','freq_ghz':3.5,'power_dbm':37,'sectors':3,'azimuth':[10,130,250],'tilt':3,'status':'active'},
    {'id':'SJK-004','name':'Kabukicho North','lat':35.6955,'lng':139.7030,'height':28,'band':'n257','freq_ghz':28.0,'power_dbm':30,'sectors':4,'azimuth':[0,90,180,270],'tilt':8,'status':'active'},
    {'id':'SBY-001','name':'Shibuya Crossing','lat':35.6595,'lng':139.7004,'height':55,'band':'n77','freq_ghz':3.7,'power_dbm':43,'sectors':3,'azimuth':[20,140,260],'tilt':5,'status':'active'},
    {'id':'SBY-002','name':'Shibuya Stream','lat':35.6565,'lng':139.7030,'height':80,'band':'n78','freq_ghz':3.5,'power_dbm':40,'sectors':3,'azimuth':[0,120,240],'tilt':4,'status':'active'},
    {'id':'SBY-003','name':'Harajuku Meiji','lat':35.6702,'lng':139.7027,'height':25,'band':'n257','freq_ghz':28.0,'power_dbm':28,'sectors':4,'azimuth':[0,90,180,270],'tilt':10,'status':'active'},
    {'id':'MNT-001','name':'Tokyo Tower Site','lat':35.6586,'lng':139.7454,'height':333,'band':'n77','freq_ghz':3.7,'power_dbm':46,'sectors':6,'azimuth':[0,60,120,180,240,300],'tilt':8,'status':'active'},
    {'id':'MNT-002','name':'Roppongi Hills','lat':35.6605,'lng':139.7292,'height':54,'band':'n77','freq_ghz':3.7,'power_dbm':40,'sectors':3,'azimuth':[15,135,255],'tilt':4,'status':'active'},
    {'id':'MNT-003','name':'Toranomon Hills','lat':35.6670,'lng':139.7495,'height':52,'band':'n78','freq_ghz':3.5,'power_dbm':40,'sectors':3,'azimuth':[0,120,240],'tilt':5,'status':'active'},
    {'id':'MNT-004','name':'Azabudai Hills','lat':35.6594,'lng':139.7370,'height':64,'band':'n257','freq_ghz':28.0,'power_dbm':33,'sectors':4,'azimuth':[45,135,225,315],'tilt':7,'status':'active'},
    {'id':'CTR-001','name':'Tokyo Station Marunouchi','lat':35.6812,'lng':139.7671,'height':50,'band':'n77','freq_ghz':3.7,'power_dbm':43,'sectors':3,'azimuth':[0,120,240],'tilt':4,'status':'active'},
    {'id':'CTR-002','name':'Ginza Chuo-dori','lat':35.6717,'lng':139.7649,'height':38,'band':'n78','freq_ghz':3.5,'power_dbm':37,'sectors':3,'azimuth':[30,150,270],'tilt':3,'status':'active'},
    {'id':'CTR-003','name':'Akihabara Electric','lat':35.6984,'lng':139.7731,'height':30,'band':'n77','freq_ghz':3.7,'power_dbm':40,'sectors':3,'azimuth':[0,120,240],'tilt':4,'status':'active'},
    {'id':'IKB-001','name':'Ikebukuro Sunshine','lat':35.7295,'lng':139.7185,'height':60,'band':'n77','freq_ghz':3.7,'power_dbm':43,'sectors':3,'azimuth':[10,130,250],'tilt':5,'status':'active'},
    {'id':'IKB-002','name':'Ikebukuro West Gate','lat':35.7310,'lng':139.7109,'height':35,'band':'n78','freq_ghz':3.5,'power_dbm':37,'sectors':3,'azimuth':[0,120,240],'tilt':3,'status':'active'},
    {'id':'SGW-001','name':'Shinagawa Station','lat':35.6284,'lng':139.7387,'height':45,'band':'n77','freq_ghz':3.7,'power_dbm':43,'sectors':3,'azimuth':[0,120,240],'tilt':4,'status':'active'},
    {'id':'ODB-001','name':'Odaiba Telecom Center','lat':35.6190,'lng':139.7760,'height':70,'band':'n77','freq_ghz':3.7,'power_dbm':46,'sectors':6,'azimuth':[0,60,120,180,240,300],'tilt':6,'status':'active'},
    {'id':'SMD-001','name':'Sumida Skytree Area','lat':35.7101,'lng':139.8107,'height':40,'band':'n78','freq_ghz':3.5,'power_dbm':40,'sectors':3,'azimuth':[0,120,240],'tilt':4,'status':'active'},
    {'id':'HO-001','name':'Yamanote Ueno Junction','lat':35.7135,'lng':139.7770,'height':22,'band':'n77','freq_ghz':3.7,'power_dbm':34,'sectors':2,'azimuth':[90,270],'tilt':2,'status':'degraded'},
    {'id':'HO-002','name':'Meguro River Corridor','lat':35.6420,'lng':139.7150,'height':18,'band':'n78','freq_ghz':3.5,'power_dbm':30,'sectors':2,'azimuth':[0,180],'tilt':2,'status':'degraded'},
    {'id':'HO-003','name':'Shuto Expressway C1 Loop','lat':35.6750,'lng':139.7550,'height':15,'band':'n77','freq_ghz':3.7,'power_dbm':33,'sectors':2,'azimuth':[45,225],'tilt':2,'status':'degraded'},
]

print(f'{len(towers)} towers | Active: {sum(1 for t in towers if t["status"]=="active")} | Degraded: {sum(1 for t in towers if t["status"]=="degraded")}')

## 4. Pull Tokyo Buildings

**Priority: PLATEAU CityGML LOD2** (MLIT open data, higher fidelity roof geometry + official heights).
**Fallback: OpenStreetMap** via Overpass API (if PLATEAU download fails on Colab).

PLATEAU LOD2 gives real measured building heights and roof shapes — critical for accurate ray-tracing. OSM heights are often estimated from floor count × 3m.

In [ ]:
import requests, json, os, time, math, zipfile, io
from shapely.geometry import Polygon, box
import geopandas as gpd
from pyproj import Transformer

# Bbox covers all 22 towers with ~1km margin
SOUTH, WEST = 35.612, 139.680
NORTH, EAST = 35.738, 139.820
REF_LAT = (SOUTH + NORTH) / 2
REF_LNG = (WEST + EAST) / 2

# UTM zone 54N (Tokyo) — proper cartographic projection
wgs2utm = Transformer.from_crs('EPSG:4326', 'EPSG:32654', always_xy=True)
utm2wgs = Transformer.from_crs('EPSG:32654', 'EPSG:4326', always_xy=True)
REF_E, REF_N = wgs2utm.transform(REF_LNG, REF_LAT)

def to_local(lat, lng):
    e, n = wgs2utm.transform(lng, lat)
    return e - REF_E, n - REF_N

def to_latlng(x, y):
    lng, lat = utm2wgs.transform(x + REF_E, y + REF_N)
    return lat, lng

print(f'Projection: UTM zone 54N (EPSG:32654)')
print(f'Origin: {REF_LAT:.4f}N, {REF_LNG:.4f}E → {REF_E:.0f}E, {REF_N:.0f}N')

# =====================================================
# PLATEAU CityGML LOD2 — try first, fall back to OSM
# =====================================================
PLATEAU_USED = False

try:
    from lxml import etree
    HAS_LXML = True
except ImportError:
    os.system('pip install -q lxml')
    try:
        from lxml import etree
        HAS_LXML = True
    except:
        HAS_LXML = False

if HAS_LXML:
    # PLATEAU Tokyo 23-ku CityGML — Shinjuku ward (13104) as pilot area
    # Full dataset: https://www.geospatial.jp/ckan/dataset/plateau-13100-tokyo23ku-2022
    # Using the 3D Tiles catalog API to find direct download URLs
    PLATEAU_CATALOG = 'https://assets.cms.plateau.reearth.io/assets/d6/702a48-dbbe-4092-8709-68df92486c87/13100_tokyo23-ku_2022_citygml_1_2_op.zip'

    plateau_dir = 'plateau_tokyo'
    os.makedirs(plateau_dir, exist_ok=True)

    gml_files = [f for f in os.listdir(plateau_dir) if f.endswith('.gml')] if os.path.exists(plateau_dir) else []

    if not gml_files:
        print('Attempting PLATEAU CityGML download (Tokyo 23-ku LOD2)...')
        print('Note: Full dataset is ~2GB. Downloading index to find relevant tiles...')
        try:
            # Try smaller per-ward download first — Shinjuku (13104)
            ward_urls = {
                '13104': 'https://assets.cms.plateau.reearth.io/assets/11/ac01c9-c5e2-4e60-a5f3-3b888ae3939b/13104_shinjuku-ku_2022_citygml_1_2_op.zip',
                '13113': 'https://assets.cms.plateau.reearth.io/assets/29/0fa5f7-cc04-4d2f-9aeb-33ebc1e7db80/13113_shibuya-ku_2022_citygml_1_2_op.zip',
            }
            for ward_code, url in ward_urls.items():
                print(f'  Trying ward {ward_code}...', end=' ', flush=True)
                resp = requests.get(url, timeout=60, stream=True)
                if resp.status_code == 200:
                    total = int(resp.headers.get('content-length', 0))
                    print(f'{total/1e6:.0f} MB', end=' ', flush=True)
                    if total > 500_000_000:  # Skip if > 500MB
                        print('(too large for Colab, skipping)')
                        continue
                    data = resp.content
                    zf = zipfile.ZipFile(io.BytesIO(data))
                    bldg_files = [n for n in zf.namelist() if '/bldg/' in n and n.endswith('.gml')]
                    for bf in bldg_files[:5]:  # Extract first 5 GML tiles
                        zf.extract(bf, plateau_dir)
                    print(f'extracted {len(bldg_files[:5])} tiles')
                else:
                    print(f'HTTP {resp.status_code}')
        except Exception as ex:
            print(f'  PLATEAU download failed: {ex}')

    # Parse whatever GML files we have
    gml_files = []
    for root_dir, dirs, files in os.walk(plateau_dir):
        for f in files:
            if f.endswith('.gml') and 'bldg' in root_dir:
                gml_files.append(os.path.join(root_dir, f))

    if gml_files:
        print(f'\nParsing {len(gml_files)} PLATEAU CityGML tiles...')
        ns = {
            'core': 'http://www.opengis.net/citygml/2.0',
            'bldg': 'http://www.opengis.net/citygml/building/2.0',
            'gml': 'http://www.opengis.net/gml',
        }
        plateau_buildings = []
        for gml_path in gml_files:
            try:
                tree = etree.parse(gml_path)
                for member in tree.findall('.//core:cityObjectMember', ns):
                    bldg = member.find('.//bldg:Building', ns)
                    if bldg is None:
                        continue
                    # Get measured height
                    mh = bldg.find('.//bldg:measuredHeight', ns)
                    height = float(mh.text) if mh is not None and mh.text else None
                    if height is None:
                        continue
                    # Get footprint from LOD2 solid or LOD0 footprint
                    pos_lists = bldg.findall('.//gml:posList', ns)
                    if not pos_lists:
                        continue
                    # Use first ground-level polygon as footprint
                    for pl in pos_lists:
                        coords_text = pl.text.strip().split()
                        if len(coords_text) < 9:
                            continue
                        coords = []
                        for i in range(0, len(coords_text) - 2, 3):
                            lat, lng = float(coords_text[i]), float(coords_text[i+1])
                            z = float(coords_text[i+2])
                            if z < 5:  # ground-level polygon
                                coords.append((lng, lat))
                        if len(coords) >= 3:
                            poly = Polygon(coords)
                            if poly.is_valid and poly.area > 0:
                                centroid = poly.centroid
                                plateau_buildings.append({
                                    'geometry': poly, 'bldg_height': height,
                                    'bldg_type': 'plateau_lod2',
                                    'centroid_lat': centroid.y, 'centroid_lng': centroid.x
                                })
                            break
            except Exception as ex:
                print(f'  Parse error in {os.path.basename(gml_path)}: {ex}')

        if plateau_buildings:
            print(f'  PLATEAU: {len(plateau_buildings)} buildings with measured heights')
            PLATEAU_USED = True
        else:
            print(f'  PLATEAU: parsed but no valid buildings extracted')

print(f'PLATEAU source: {"YES ✓" if PLATEAU_USED else "NO — falling back to OSM"}')

# =====================================================
# OpenStreetMap fallback (always runs to fill gaps)
# =====================================================
URL = 'https://overpass.kumi.systems/api/interpreter'
n_lat, n_lng = 5, 5
lat_step = (NORTH - SOUTH) / n_lat
lng_step = (EAST - WEST) / n_lng
all_nodes = {}
all_ways = []

print(f'\nPulling OSM buildings ({n_lat*n_lng} tiles)...')
for i in range(n_lat):
    for j in range(n_lng):
        s = SOUTH + i * lat_step
        n = SOUTH + (i+1) * lat_step
        w = WEST + j * lng_step
        e = WEST + (j+1) * lng_step
        tile = f'{i*n_lng+j+1}/{n_lat*n_lng}'
        query = f'[out:json][timeout:90];way["building"]({s},{w},{n},{e});out body;>;out skel qt;'
        for attempt in range(3):
            try:
                print(f'  Tile {tile}...', end=' ', flush=True)
                resp = requests.post(URL, data={'data': query}, timeout=120,
                    headers={'User-Agent': 'WINNIIO-DT/1.0'})
                if resp.status_code != 200:
                    raise Exception(f'HTTP {resp.status_code}')
                data = resp.json()
                nw = sum(1 for el in data['elements'] if el['type'] == 'way' and 'tags' in el)
                print(f'{nw} buildings')
                for el in data['elements']:
                    if el['type'] == 'node':
                        all_nodes[el['id']] = (el['lat'], el['lon'])
                    elif el['type'] == 'way' and 'tags' in el:
                        all_ways.append(el)
                break
            except Exception as ex:
                print(f'retry ({ex})...')
                time.sleep(5)
        time.sleep(1)

osm_buildings = []
for el in all_ways:
    if 'building' not in el.get('tags', {}): continue
    coords = [(all_nodes[nid][1], all_nodes[nid][0]) for nid in el.get('nodes', []) if nid in all_nodes]
    if len(coords) < 4: continue
    tags = el['tags']
    height = 10.0
    if 'height' in tags:
        try: height = float(str(tags['height']).replace('m', '').strip())
        except ValueError: pass
    elif 'building:levels' in tags:
        try: height = float(tags['building:levels']) * 3.0
        except ValueError: pass
    poly = Polygon(coords)
    centroid = poly.centroid
    osm_buildings.append({'geometry': poly, 'bldg_height': height,
                          'bldg_type': tags.get('building', 'yes'),
                          'centroid_lat': centroid.y, 'centroid_lng': centroid.x})

# Merge: PLATEAU buildings take priority, OSM fills gaps
import pandas as pd
if PLATEAU_USED:
    all_buildings = plateau_buildings + osm_buildings
    print(f'\nMerged: {len(plateau_buildings)} PLATEAU + {len(osm_buildings)} OSM = {len(all_buildings)} total')
else:
    all_buildings = osm_buildings
    print(f'\n{len(osm_buildings)} OSM buildings')

gdf = gpd.GeoDataFrame(all_buildings, geometry='geometry')
gdf = gdf[gdf.geometry.is_valid].copy()

# --- DEM ---
try:
    import srtm
    elev_data = srtm.get_data()
    def get_elevation(lat, lng):
        e = elev_data.get_elevation(lat, lng)
        return float(e) if e is not None else 0.0
    gdf['ground_elev'] = gdf.apply(lambda r: get_elevation(r['centroid_lat'], r['centroid_lng']), axis=1)
    elev_min = gdf['ground_elev'].min()
    gdf['ground_elev'] -= elev_min
    print(f'DEM loaded. Elevation: {gdf["ground_elev"].min():.0f}m — {gdf["ground_elev"].max():.0f}m')
    HAS_DEM = True
except Exception as ex:
    print(f'DEM unavailable ({ex}) — flat terrain')
    gdf['ground_elev'] = 0.0
    HAS_DEM = False

# --- Spatial selection ---
tower_points = gpd.GeoDataFrame(
    geometry=[box(t['lng']-0.005, t['lat']-0.005, t['lng']+0.005, t['lat']+0.005) for t in towers],
    crs='EPSG:4326'
)
near_towers = gpd.sjoin(gdf, tower_points, how='inner', predicate='intersects').drop_duplicates(subset=gdf.columns)
far_from_towers = gdf[~gdf.index.isin(near_towers.index)]

MAX_BUILDINGS = 4000
nearby = near_towers.nlargest(min(len(near_towers), MAX_BUILDINGS), 'bldg_height')
remaining_slots = MAX_BUILDINGS - len(nearby)
if remaining_slots > 0:
    filler = far_from_towers.nlargest(remaining_slots, 'bldg_height')
    sample = gpd.GeoDataFrame(pd.concat([nearby, filler], ignore_index=True))
else:
    sample = nearby

commercial_types = {'commercial', 'office', 'retail', 'hotel', 'apartments', 'residential'}
sample['is_commercial'] = sample['bldg_type'].isin(commercial_types) | (sample['bldg_height'] > 30)

n_plateau = (sample['bldg_type'] == 'plateau_lod2').sum() if 'bldg_type' in sample.columns else 0
print(f'\n{len(sample)} buildings selected')
if n_plateau > 0:
    print(f'  PLATEAU LOD2: {n_plateau} | OSM: {len(sample) - n_plateau}')
print(f'  Glass: {sample["is_commercial"].sum()} | Concrete: {(~sample["is_commercial"]).sum()}')
print(f'  Height: {sample["bldg_height"].min():.0f}m — {sample["bldg_height"].max():.0f}m')

## 5. Convert to PLY + Mitsuba XML with ITU Radio Materials

PLY is the mesh format Sionna expects. ITU-R P.2040 materials give physically accurate RF reflection/refraction.

In [ ]:
import struct, pandas as pd

CACHE_DIR = 'tokyo_scene/mesh'
os.makedirs(CACHE_DIR, exist_ok=True)

concrete_ply = f'{CACHE_DIR}/buildings_concrete.ply'
glass_ply = f'{CACHE_DIR}/buildings_glass.ply'
ground_ply = f'{CACHE_DIR}/ground.ply'

# Check cache — skip mesh generation if PLY files already exist
if os.path.exists(concrete_ply) and os.path.exists(ground_ply):
    print(f'PLY cache hit — skipping mesh generation')
    print(f'  {concrete_ply} ({os.path.getsize(concrete_ply)/1e6:.1f} MB)')
    if os.path.exists(glass_ply):
        print(f'  {glass_ply} ({os.path.getsize(glass_ply)/1e6:.1f} MB)')
    print(f'  {ground_ply}')
    print('Delete tokyo_scene/mesh/ to force rebuild.')
else:
    def build_ply(rows, ply_path):
        """Convert GeoDataFrame rows to PLY mesh with terrain elevation."""
        all_vertices = []
        all_faces = []
        vertex_offset = 0
        for _, row in rows.iterrows():
            geom = row.geometry
            if geom.geom_type == 'MultiPolygon':
                geom = list(geom.geoms)[0]
            coords = list(geom.exterior.coords)[:-1]
            if len(coords) < 3:
                continue
            h = row['bldg_height']
            ground = row.get('ground_elev', 0.0)
            n = len(coords)
            for c in coords:
                x, y = to_local(c[1], c[0])
                all_vertices.append((x, ground, y))          # base at terrain
                all_vertices.append((x, ground + h, y))      # top at terrain + height
            for i in range(n):
                j = (i + 1) % n
                b1 = vertex_offset + i * 2
                t1 = vertex_offset + i * 2 + 1
                b2 = vertex_offset + j * 2
                t2 = vertex_offset + j * 2 + 1
                all_faces.append((b1, b2, t2))
                all_faces.append((b1, t2, t1))
            top_base = vertex_offset + 1
            for i in range(1, n - 1):
                all_faces.append((top_base, top_base + i * 2, top_base + (i + 1) * 2))
            vertex_offset += n * 2
        with open(ply_path, 'w') as f:
            f.write('ply\nformat ascii 1.0\n')
            f.write(f'element vertex {len(all_vertices)}\n')
            f.write('property float x\nproperty float y\nproperty float z\n')
            f.write(f'element face {len(all_faces)}\n')
            f.write('property list uchar int vertex_indices\n')
            f.write('end_header\n')
            for v in all_vertices:
                f.write(f'{v[0]:.2f} {v[1]:.2f} {v[2]:.2f}\n')
            for face in all_faces:
                f.write(f'3 {face[0]} {face[1]} {face[2]}\n')
        return len(all_vertices), len(all_faces)

    # Split buildings: concrete vs glass
    concrete_bldgs = sample[~sample['is_commercial']]
    glass_bldgs = sample[sample['is_commercial']]

    nv, nf = build_ply(concrete_bldgs, concrete_ply)
    print(f'Concrete PLY: {nv} vertices, {nf} faces ({os.path.getsize(concrete_ply)/1e6:.1f} MB)')

    if len(glass_bldgs) > 0:
        nv2, nf2 = build_ply(glass_bldgs, glass_ply)
        print(f'Glass PLY: {nv2} vertices, {nf2} faces ({os.path.getsize(glass_ply)/1e6:.1f} MB)')

    # Ground plane with DEM elevation grid
    S = 10000
    if HAS_DEM:
        # Build terrain mesh from DEM — 200m grid
        grid_step = 200
        grid_n = int(2 * S / grid_step) + 1
        xs = np.linspace(-S, S, grid_n)
        ys = np.linspace(-S, S, grid_n)
        ground_verts = []
        for yi in ys:
            for xi in xs:
                lat, lng = to_latlng(xi, yi)
                try:
                    elev = get_elevation(lat, lng) - elev_min
                except:
                    elev = 0.0
                ground_verts.append((xi, elev, yi))
        ground_faces = []
        for r in range(grid_n - 1):
            for c in range(grid_n - 1):
                v0 = r * grid_n + c
                v1 = v0 + 1
                v2 = v0 + grid_n
                v3 = v2 + 1
                ground_faces.append((v0, v1, v3))
                ground_faces.append((v0, v3, v2))
        with open(ground_ply, 'w') as f:
            f.write('ply\nformat ascii 1.0\n')
            f.write(f'element vertex {len(ground_verts)}\n')
            f.write('property float x\nproperty float y\nproperty float z\n')
            f.write(f'element face {len(ground_faces)}\n')
            f.write('property list uchar int vertex_indices\n')
            f.write('end_header\n')
            for v in ground_verts:
                f.write(f'{v[0]:.2f} {v[1]:.2f} {v[2]:.2f}\n')
            for face in ground_faces:
                f.write(f'3 {face[0]} {face[1]} {face[2]}\n')
        print(f'Terrain PLY: {len(ground_verts)} vertices, {len(ground_faces)} faces (DEM grid {grid_step}m)')
    else:
        with open(ground_ply, 'w') as f:
            f.write('ply\nformat ascii 1.0\n')
            f.write('element vertex 4\n')
            f.write('property float x\nproperty float y\nproperty float z\n')
            f.write('element face 2\n')
            f.write('property list uchar int vertex_indices\n')
            f.write('end_header\n')
            f.write(f'{-S} 0.0 {-S}\n{S} 0.0 {-S}\n{S} 0.0 {S}\n{-S} 0.0 {S}\n')
            f.write('3 0 1 2\n3 0 2 3\n')
        print(f'Flat ground PLY (no DEM)')

    print(f'\nAll PLY files cached in {CACHE_DIR}/ — delete to rebuild')

In [ ]:
# Mitsuba 3 XML with ITU radio materials — concrete + glass + wet_ground
scene_xml = """<?xml version="1.0" encoding="utf-8"?>
<scene version="3.0.0">
    <integrator type="path"/>

    <!-- ITU-R P.2040 radio materials for physically accurate RF propagation -->
    <bsdf type="itu-radio-material" id="mat-itu_concrete">
        <string name="type" value="concrete"/>
        <float name="thickness" value="0.15"/>
    </bsdf>

    <bsdf type="itu-radio-material" id="mat-itu_glass">
        <string name="type" value="glass"/>
        <float name="thickness" value="0.01"/>
    </bsdf>

    <bsdf type="itu-radio-material" id="mat-itu_wet_ground">
        <string name="type" value="wet_ground"/>
    </bsdf>

    <!-- Ground plane -->
    <shape type="ply" id="ground">
        <string name="filename" value="mesh/ground.ply"/>
        <ref name="bsdf" id="mat-itu_wet_ground"/>
        <boolean name="face_normals" value="true"/>
    </shape>

    <!-- Concrete buildings (residential, low-rise) -->
    <shape type="ply" id="buildings_concrete">
        <string name="filename" value="mesh/buildings_concrete.ply"/>
        <ref name="bsdf" id="mat-itu_concrete"/>
        <boolean name="face_normals" value="true"/>
    </shape>

    <!-- Glass-facade buildings (commercial, office, tall) -->
    <shape type="ply" id="buildings_glass">
        <string name="filename" value="mesh/buildings_glass.ply"/>
        <ref name="bsdf" id="mat-itu_glass"/>
        <boolean name="face_normals" value="true"/>
    </shape>
</scene>
"""

scene_path = 'tokyo_scene/scene.xml'
with open(scene_path, 'w') as f:
    f.write(scene_xml)

print(f'Mitsuba scene: {scene_path}')
print('Materials: concrete (walls), glass (commercial facades), wet_ground (terrain)')

## 6. Custom Antenna Patterns + Scene Load

Realistic macro cell antenna: 65° horizontal beamwidth, 7° vertical beamwidth, ~18 dBi gain. Based on typical Kathrein/CommScope specs used on Rakuten towers. In Phase 1, replace with actual vendor datasheets from Altiostar.

In [ ]:
scene = load_scene('tokyo_scene/scene.xml')

# --- Custom macro cell antenna pattern ---
# Approximation of typical 65° HPBW panel (Kathrein 742 265 / CommScope HHVHX310R2)
# In Phase 1: replace with actual Altiostar vendor antenna files
import torch

def macro_sector_pattern(theta, phi):
    """3GPP TR 38.901 Table 7.3-1 antenna model with realistic parameters.
    theta: zenith angle (0=boresight up, pi/2=horizon)
    phi: azimuth angle
    Returns: antenna gain pattern (linear scale)
    """
    # Horizontal beamwidth 65°, vertical beamwidth 7°
    phi_3db = np.radians(65)
    theta_3db = np.radians(7)
    Am = 30.0   # front-to-back ratio (dB)
    SLA_v = 30.0  # side-lobe level limit (dB)
    G_max = 18.0  # peak gain (dBi) — typical for 4x4 MIMO panel

    # Convert to numpy for computation
    theta_np = theta.cpu().numpy() if isinstance(theta, torch.Tensor) else np.asarray(theta)
    phi_np = phi.cpu().numpy() if isinstance(phi, torch.Tensor) else np.asarray(phi)

    # Horizontal cut: -min(12*(phi/phi_3dB)^2, Am)
    A_h = -np.minimum(12.0 * (phi_np / phi_3db)**2, Am)

    # Vertical cut: -min(12*((theta - pi/2)/theta_3dB)^2, SLA_v)
    A_v = -np.minimum(12.0 * ((theta_np - np.pi/2) / theta_3db)**2, SLA_v)

    # Combined: -min(-(A_h + A_v), Am) + G_max
    A_db = -np.minimum(-(A_h + A_v), Am) + G_max

    # Convert to linear (field magnitude, not power)
    gain_linear = 10.0 ** (A_db / 20.0)

    result = torch.tensor(gain_linear, dtype=theta.dtype if isinstance(theta, torch.Tensor) else torch.float32)
    return result

# Use 4x4 MIMO panel with custom pattern
scene.tx_array = PlanarArray(
    num_rows=4, num_cols=4,
    vertical_spacing=0.7, horizontal_spacing=0.5,
    pattern=macro_sector_pattern, polarization='VH'
)

# UE: single dipole (standard)
scene.rx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='dipole', polarization='cross'
)

print(f'Custom antenna: 65° HPBW, 7° vertical, 18 dBi, 30 dB F/B ratio')
print(f'⚠️  Approximation — replace with actual vendor data in Phase 1')

# Place sectorized transmitters with terrain-adjusted height
tx_count = 0
for t in towers:
    x, y = to_local(t['lat'], t['lng'])
    ground = 0.0
    if HAS_DEM:
        try:
            ground = get_elevation(t['lat'], t['lng']) - elev_min
        except:
            pass
    tower_top = ground + t['height']
    for s_idx, az_deg in enumerate(t['azimuth']):
        tx = Transmitter(
            name=f"{t['id']}-S{s_idx}",
            position=[float(x), float(tower_top), float(y)],
            orientation=[float(np.radians(az_deg)), float(np.radians(t['tilt'])), 0.0],
            power_dbm=float(t['power_dbm']),
        )
        scene.add(tx)
        tx_count += 1

print(f'{tx_count} sector transmitters placed across {len(towers)} sites')
if HAS_DEM:
    print(f'Tower heights adjusted for terrain elevation (SRTM 30m DEM)')

## 7. Compute Coverage Maps — n77 (3.7 GHz)

In [ ]:
rm_solver = RadioMapSolver()

# --- n77 band (3.7 GHz) ---
scene.frequency = 3.7e9
print(f'Computing n77 coverage at {scene.frequency/1e9} GHz...')
print(f'Diffraction: ON | Samples: 10^6 | Grid: 10m')

rm_n77 = rm_solver(
    scene=scene,
    max_depth=5,
    diffraction=True,
    cell_size=[10, 10],
    samples_per_tx=10**6,
)

print('n77 coverage computed.')
rm_n77.show(metric='path_gain')

## 8. TX Association (Best Server Map)

This is the MRO-critical view: which cell serves each location? Handover boundaries are where two cells have similar RSS.

In [ ]:
# Best server association — shows handover boundaries
rm_n77.show_association(metric='rss')

## 9. Compute n78 (3.5 GHz) + n257 (28 GHz mmWave)

In [ ]:
# --- n78 band (3.5 GHz) ---
scene.frequency = 3.5e9
print(f'Computing n78 coverage at {scene.frequency/1e9} GHz...')
print(f'Diffraction: ON | Samples: 10^6 | Grid: 10m')
rm_n78 = rm_solver(
    scene=scene,
    max_depth=5,
    diffraction=True,
    cell_size=[10, 10],
    samples_per_tx=10**6,
)
print('n78 done.')
rm_n78.show(metric='path_gain')

In [ ]:
# --- n257 mmWave (28 GHz) ---
# wet_ground + glass materials not valid at 28 GHz — reload scene with concrete for both
scene_xml_path = 'tokyo_scene/scene.xml'
with open(scene_xml_path) as f:
    xml = f.read()
xml_mmw = xml.replace('value="wet_ground"', 'value="concrete"').replace('value="glass"', 'value="concrete"')
mmw_scene_path = 'tokyo_scene/scene_mmw.xml'
with open(mmw_scene_path, 'w') as f:
    f.write(xml_mmw)

scene_mmw = load_scene(mmw_scene_path)
scene_mmw.frequency = 28e9
scene_mmw.tx_array = PlanarArray(num_rows=4, num_cols=4, vertical_spacing=0.7, horizontal_spacing=0.5, pattern='tr38901', polarization='VH')
scene_mmw.rx_array = PlanarArray(num_rows=1, num_cols=1, vertical_spacing=0.5, horizontal_spacing=0.5, pattern='dipole', polarization='cross')

tx_count = 0
for t in towers:
    x, y = to_local(t['lat'], t['lng'])
    ground = 0.0
    if HAS_DEM:
        try:
            ground = get_elevation(t['lat'], t['lng']) - elev_min
        except:
            pass
    tower_top = ground + t['height']
    for s_idx, az_deg in enumerate(t['azimuth']):
        tx = Transmitter(
            name=f"{t['id']}-S{s_idx}",
            position=[float(x), float(tower_top), float(y)],
            orientation=[float(np.radians(az_deg)), float(np.radians(t['tilt'])), 0.0],
            power_dbm=float(t['power_dbm']),
        )
        scene_mmw.add(tx)
        tx_count += 1

print(f'Computing n257 coverage at 28.0 GHz (mmWave)... {tx_count} TXs')
print(f'Diffraction: ON | Samples: 2x10^6 | Grid: 5m')
rm_mmw = rm_solver(
    scene=scene_mmw,
    max_depth=5,
    diffraction=True,
    cell_size=[5, 5],
    samples_per_tx=2*10**6,
)
print('n257 done.')
rm_mmw.show(metric='path_gain')

## 10. SINR + CDF Analysis

In [ ]:
import matplotlib.pyplot as plt

# SINR map — shows interference-limited areas (HO failure candidates)
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

for ax, rm, label in [
    (axes[0], rm_n77, 'n77 (3.7 GHz)'),
    (axes[1], rm_n78, 'n78 (3.5 GHz)'),
    (axes[2], rm_mmw, 'n257 (28 GHz)'),
]:
    sinr = rm.sinr.numpy()
    # sinr shape: (num_tx, rows, cols) — take max SINR across all TXs
    sinr_combined = sinr.max(axis=0).squeeze()
    sinr_db = 10 * np.log10(np.clip(sinr_combined, 1e-10, None))
    im = ax.imshow(sinr_db, cmap='RdYlGn', vmin=-10, vmax=30,
                   origin='lower', aspect='equal')
    ax.set_title(f'SINR — {label}', fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, label='SINR (dB)', shrink=0.8)

plt.suptitle('WINNIIO Tokyo MRO — SINR by Band (Sionna Ray-Tracing)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('tokyo_sinr_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CDF: what % of area has SINR above threshold?
fig, ax = plt.subplots(figsize=(10, 6))

for rm, label, color in [
    (rm_n77, 'n77 (3.7 GHz)', '#00b4d8'),
    (rm_n78, 'n78 (3.5 GHz)', '#00e676'),
    (rm_mmw, 'n257 (28 GHz)', '#e040fb'),
]:
    sinr = rm.sinr.numpy()
    sinr_combined = sinr.max(axis=0).squeeze().flatten()
    sinr_db = 10 * np.log10(np.clip(sinr_combined[sinr_combined > 0], 1e-10, None))
    sorted_sinr = np.sort(sinr_db)
    cdf = np.arange(1, len(sorted_sinr) + 1) / len(sorted_sinr)
    ax.plot(sorted_sinr, cdf, label=label, color=color, linewidth=2)

ax.axvline(x=0, color='red', linestyle='--', alpha=0.5, label='0 dB (cell edge)')
ax.axvline(x=5, color='orange', linestyle='--', alpha=0.5, label='5 dB (HO threshold)')
ax.set_xlabel('SINR (dB)', fontsize=12)
ax.set_ylabel('CDF', fontsize=12)
ax.set_title('SINR CDF — Handover Decision Boundaries', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(-20, 40)
plt.tight_layout()
plt.savefig('tokyo_sinr_cdf.png', dpi=150)
plt.show()

print('\nArea with SINR < 0 dB (likely HO failure zone):')
for rm, label in [(rm_n77, 'n77'), (rm_n78, 'n78'), (rm_mmw, 'n257')]:
    sinr = rm.sinr.numpy()
    sinr_combined = sinr.max(axis=0).squeeze().flatten()
    sinr_db = 10 * np.log10(np.clip(sinr_combined[sinr_combined > 0], 1e-10, None))
    pct_bad = (sinr_db < 0).sum() / len(sinr_db) * 100
    print(f'  {label}: {pct_bad:.1f}% of coverage area')

## 11. 3D Scene Preview with Coverage Overlay

In [ ]:
# 3D rendering with radio map overlay
scene.frequency = 3.7e9  # reset to n77
scene.preview(radio_map=rm_n77, rm_metric='path_gain')

## 12. Export GeoJSON for Cesium Demo Overlay

In [ ]:
import geojson

def export_radiomap_geojson(rm, output_path, step=4):
    """Export Sionna radio map as GeoJSON for Cesium overlay."""
    rss = rm.rss.numpy()
    # rss shape: (num_tx, rows, cols) — take max across TXs (best server)
    rss_best = rss.max(axis=0).squeeze()
    rss_dbm = 10 * np.log10(np.clip(rss_best, 1e-30, None)) + 30
    rows, cols = rss_dbm.shape
    x_min = -((cols * 10) / 2)
    y_min = -((rows * 10) / 2)
    features = []
    for r in range(0, rows, step):
        for c in range(0, cols, step):
            val = float(rss_dbm[r, c])
            if val < -130:
                continue
            x = x_min + c * 10
            y = y_min + r * 10
            lat, lng = to_latlng(x, y)
            if val > -80:
                color, quality = '#00e676', 'strong'
            elif val > -95:
                color, quality = '#ffca28', 'moderate'
            else:
                color, quality = '#ff5252', 'weak'
            half = step * 10 / 2
            lat1, lng1 = to_latlng(x - half, y - half)
            lat2, lng2 = to_latlng(x + half, y + half)
            features.append(geojson.Feature(
                geometry=geojson.Polygon([[
                    [lng1, lat1], [lng2, lat1], [lng2, lat2], [lng1, lat2], [lng1, lat1]
                ]]),
                properties={'rss_dbm': round(val, 1), 'quality': quality,
                            'fill': color, 'fill-opacity': 0.4}
            ))
    with open(output_path, 'w') as f:
        geojson.dump(geojson.FeatureCollection(features), f)
    print(f'Exported {len(features)} cells to {output_path}')

export_radiomap_geojson(rm_n77, 'tokyo_coverage_n77.geojson', step=3)
export_radiomap_geojson(rm_n78, 'tokyo_coverage_n78.geojson', step=3)
export_radiomap_geojson(rm_mmw, 'tokyo_coverage_n257.geojson', step=2)

from google.colab import files
files.download('tokyo_coverage_n77.geojson')
files.download('tokyo_coverage_n78.geojson')
files.download('tokyo_coverage_n257.geojson')
files.download('tokyo_sinr_comparison.png')
files.download('tokyo_sinr_cdf.png')
print('All files exported.')

## 12b. Export Cesium-Native Coverage GeoJSON

Exports the Sionna ray-traced RSRP grid as a GeoJSON that the Cesium demo (`altiostar-tokyo-demo.html`) loads directly via `Cesium.GeoJsonDataSource`. Each grid cell becomes a polygon with RSRP value and color — so the 3D coverage pillars in the demo reflect **actual building shadows**, not synthetic data.

Run this cell, download the 3 GeoJSON files, and place them alongside the HTML file. The Cesium demo's Sionna RT toggle will load these instead of the synthetic overlay.

In [ ]:
# --- Export Sionna coverage as Cesium-native GeoJSON ---
# Each grid cell → lat/lng polygon with RSRP, color, and extrusion height.
# The Cesium demo loads these directly — no synthetic overlay needed.

import geojson as gj

def export_cesium_coverage(rm, band_label, output_path, cell_size_m=10, step=3):
    """Export Sionna radio map as Cesium-ready GeoJSON with extrusion heights.
    
    Each polygon gets:
      - rsrp_dbm: actual ray-traced value
      - fill/stroke: color by signal quality
      - extrude_height: proportional to signal strength (for 3D pillars)
    """
    rss = rm.rss.numpy()
    rss_best = rss.max(axis=0).squeeze()
    rss_dbm = 10 * np.log10(np.clip(rss_best, 1e-30, None)) + 30
    rows, cols = rss_dbm.shape
    
    x_min = -((cols * cell_size_m) / 2)
    y_min = -((rows * cell_size_m) / 2)
    
    features = []
    for r in range(0, rows, step):
        for c in range(0, cols, step):
            val = float(rss_dbm[r, c])
            if val < -130:
                continue  # no coverage — skip
            
            # Grid cell center in local coords
            x = x_min + c * cell_size_m
            y = y_min + r * cell_size_m
            
            # Color by RSRP quality
            if val > -75:
                color = '#00e676'   # strong — green
                quality = 'strong'
            elif val > -90:
                color = '#66bb6a'   # good — light green
                quality = 'good'
            elif val > -105:
                color = '#ffca28'   # moderate — yellow/orange
                quality = 'moderate'
            elif val > -115:
                color = '#ff9800'   # weak — orange
                quality = 'weak'
            else:
                color = '#ff5252'   # very weak — red
                quality = 'very_weak'
            
            # Extrusion height: scale RSRP to visual height (0-400m)
            # -130 dBm → 0m, -60 dBm → 400m
            extrude = max(0, (val + 130) / 70 * 400)
            
            # Convert grid cell corners to lat/lng
            half = step * cell_size_m / 2
            lat_sw, lng_sw = to_latlng(x - half, y - half)
            lat_ne, lng_ne = to_latlng(x + half, y + half)
            
            features.append(gj.Feature(
                geometry=gj.Polygon([[
                    [lng_sw, lat_sw],
                    [lng_ne, lat_sw],
                    [lng_ne, lat_ne],
                    [lng_sw, lat_ne],
                    [lng_sw, lat_sw],
                ]]),
                properties={
                    'rsrp_dbm': round(val, 1),
                    'quality': quality,
                    'band': band_label,
                    'fill': color,
                    'fill-opacity': 0.18,
                    'stroke': color,
                    'stroke-opacity': 0.3,
                    'stroke-width': 0,
                    'extrude_height': round(extrude, 1),
                }
            ))
    
    fc = gj.FeatureCollection(features)
    with open(output_path, 'w') as f:
        gj.dump(fc, f)
    
    # Stats
    qualities = {}
    for feat in features:
        q = feat['properties']['quality']
        qualities[q] = qualities.get(q, 0) + 1
    
    print(f'\n{band_label}: {len(features)} cells → {output_path}')
    print(f'  File size: {os.path.getsize(output_path)/1e6:.1f} MB')
    for q in ['strong', 'good', 'moderate', 'weak', 'very_weak']:
        if q in qualities:
            pct = qualities[q] / len(features) * 100
            print(f'  {q}: {qualities[q]} ({pct:.0f}%)')
    return output_path

# Export all three bands
print('Exporting Sionna coverage for Cesium demo...')
print('These files replace the synthetic overlay — coverage matches building shadows.')

f1 = export_cesium_coverage(rm_n77, 'n77', 'cesium_coverage_n77.geojson', cell_size_m=10, step=3)
f2 = export_cesium_coverage(rm_n78, 'n78', 'cesium_coverage_n78.geojson', cell_size_m=10, step=3)
f3 = export_cesium_coverage(rm_mmw, 'n257', 'cesium_coverage_n257.geojson', cell_size_m=5, step=2)

# Download for local use with Cesium demo
try:
    from google.colab import files
    files.download(f1)
    files.download(f2)
    files.download(f3)
    print('\n✓ Downloaded — place these files next to altiostar-tokyo-demo.html')
except:
    print(f'\nFiles saved locally. Copy to your Cesium demo directory.')

print('\nCesium demo integration:')
print('  1. Place cesium_coverage_*.geojson alongside the HTML file')
print('  2. The demo loads them via Cesium.GeoJsonDataSource')
print('  3. Toggle "Sionna RT" button → coverage from real ray-tracing')
print('  4. Building shadows visible because coverage was computed FROM buildings')

## 13. Calibration Stub — MDT/Drive Test Import

When Altiostar provides Minimization of Drive Test (MDT) data or drive test logs, this cell calibrates the simulation against real measurements. **Phase 1 deliverable: RMSE < 8 dB between simulated and measured RSRP.**

In [ ]:
# --- Calibration stub: swap in real MDT data from Altiostar ---
# Expected CSV format: lat, lng, rsrp_dbm, band, timestamp
# This cell compares simulated vs measured RSRP and computes calibration error

import pandas as pd

# Placeholder: generate synthetic MDT points for demo
# In Phase 1, replace with: mdt = pd.read_csv('altiostar_mdt_tokyo.csv')
np.random.seed(42)
n_mdt = 200
mdt_lats = np.random.uniform(SOUTH + 0.01, NORTH - 0.01, n_mdt)
mdt_lngs = np.random.uniform(WEST + 0.01, EAST - 0.01, n_mdt)
mdt_rsrp = np.random.uniform(-110, -60, n_mdt)  # synthetic — REPLACE WITH REAL DATA
mdt = pd.DataFrame({'lat': mdt_lats, 'lng': mdt_lngs, 'rsrp_measured': mdt_rsrp})

# Sample simulated RSRP at MDT locations
rss = rm_n77.rss.numpy()
rss_best = rss.max(axis=0).squeeze()
rss_dbm = 10 * np.log10(np.clip(rss_best, 1e-30, None)) + 30
rows, cols = rss_dbm.shape

sim_rsrp = []
for _, row in mdt.iterrows():
    x, y = to_local(row['lat'], row['lng'])
    x_min = -((cols * 10) / 2)
    y_min = -((rows * 10) / 2)
    c = int((x - x_min) / 10)
    r = int((y - y_min) / 10)
    if 0 <= r < rows and 0 <= c < cols:
        sim_rsrp.append(float(rss_dbm[r, c]))
    else:
        sim_rsrp.append(np.nan)

mdt['rsrp_simulated'] = sim_rsrp
mdt_valid = mdt.dropna()

rmse = np.sqrt(np.mean((mdt_valid['rsrp_measured'] - mdt_valid['rsrp_simulated'])**2))
mae = np.mean(np.abs(mdt_valid['rsrp_measured'] - mdt_valid['rsrp_simulated']))
bias = np.mean(mdt_valid['rsrp_simulated'] - mdt_valid['rsrp_measured'])

print(f'=== Calibration Report (SYNTHETIC DATA — replace with real MDT) ===')
print(f'  Points: {len(mdt_valid)} / {len(mdt)}')
print(f'  RMSE:   {rmse:.1f} dB')
print(f'  MAE:    {mae:.1f} dB')
print(f'  Bias:   {bias:+.1f} dB (positive = sim overestimates)')
print(f'  Target: RMSE < 8 dB (typical for calibrated ray-tracing)')
print(f'\n⚠️  These numbers are meaningless until real MDT data is loaded.')

## 14. RL Hook Skeleton — MRO Agent (Phase 2+)

Reinforcement learning stub for Mobility Robustness Optimization. The agent learns optimal CIO/handover parameters from the Sionna-simulated environment. **This is the production path: train in simulation, deploy as xApp on near-RT RIC.**

In [ ]:
# --- RL Hook: MRO Environment Skeleton ---
# Phase 2+: Connect Sionna coverage to Stable Baselines3 RL training
# This stub defines the reward function and action/observation spaces

class MROEnvironmentStub:
    """Skeleton for MRO RL environment using Sionna coverage maps."""

    def __init__(self, towers, coverage_map):
        self.towers = towers
        self.coverage = coverage_map
        self.n_cells = sum(t['sectors'] for t in towers)

        # Action space: CIO adjustment per cell pair (-6 to +6 dB, step 1)
        # In production: gym.spaces.MultiDiscrete
        self.action_dim = self.n_cells * (self.n_cells - 1)

        # Observation space: RSRP, SINR, load per cell + UE distribution
        self.obs_dim = self.n_cells * 4  # [rsrp_avg, sinr_avg, load, n_ue]

    def reward(self, ho_success, ho_failure, ho_pingpong, ho_late):
        """MRO reward function aligned with Altiostar's requirements."""
        r_success = ho_success * 1.0
        r_failure = ho_failure * -10.0
        r_pingpong = ho_pingpong * -5.0
        r_late = ho_late * -3.0
        return r_success + r_failure + r_pingpong + r_late

    def step(self, action):
        """Apply CIO changes, re-simulate coverage, compute reward."""
        # Phase 2: action modifies tower CIO values
        # Re-run Sionna with updated parameters
        # Count HO events from TX association changes
        raise NotImplementedError('Connect to Sionna scene in Phase 2')

    def get_ho_boundaries(self, rm):
        """Extract handover boundary pixels from TX association map."""
        # Pixels where best-server and second-best-server RSS differ < 3 dB
        rss = rm.rss.numpy()
        rss_sorted = np.sort(rss, axis=0)
        best = rss_sorted[-1]
        second = rss_sorted[-2]
        margin_db = 10 * np.log10(np.clip(best / (second + 1e-30), 1e-10, None))
        return margin_db < 3.0  # boolean mask of HO boundary pixels

env = MROEnvironmentStub(towers, rm_n77)
print(f'MRO RL Environment:')
print(f'  Cells: {env.n_cells}')
print(f'  Action dim: {env.action_dim} (CIO adjustments)')
print(f'  Obs dim: {env.obs_dim}')
print(f'  Reward: +1 success, -10 failure, -5 pingpong, -3 late HO')
print(f'\n⚠️  Stub only — connect to Stable Baselines3 in Phase 2')
print(f'    pip install stable-baselines3')
print(f'    model = PPO("MlpPolicy", env, verbose=1)')
print(f'    model.learn(total_timesteps=100_000)')

## 15. UE Mobility Simulation — Handover Trace

Simulates UE (User Equipment) movement along realistic routes through the Tokyo tower network. At each waypoint, samples RSRP from the coverage map to determine serving cell and detect handover events (success, failure, ping-pong). Exports trace as GeoJSON for Cesium overlay.

**This is the core MRO input:** without UE trajectories, you can't compute handover statistics. In Phase 1, replace with real MDT/CHR traces from Altiostar.

In [ ]:
# --- UE Mobility Simulation with RSRP Sampling ---
# Generates realistic UE routes between tower clusters, samples RSRP at each point,
# detects serving cell changes (handovers), and classifies HO events.

import geojson as gj

# --- Route generation: realistic paths through the network ---
# 5 routes covering different mobility scenarios
routes = [
    {
        'name': 'Yamanote Line (Shibuya→Shinjuku→Ikebukuro)',
        'type': 'train',
        'speed_kmh': 40,
        'waypoints': [
            (35.6595, 139.7004),  # Shibuya
            (35.6620, 139.6995),
            (35.6702, 139.7027),  # Harajuku
            (35.6750, 139.7010),
            (35.6830, 139.6990),
            (35.6896, 139.6982),  # Shinjuku
            (35.6935, 139.6917),  # Nishi-Shinjuku
            (35.6980, 139.6950),
            (35.7050, 139.6990),
            (35.7120, 139.7050),
            (35.7200, 139.7100),
            (35.7295, 139.7185),  # Ikebukuro
        ],
    },
    {
        'name': 'Shuto Expressway C1 (high speed)',
        'type': 'vehicle',
        'speed_kmh': 80,
        'waypoints': [
            (35.6586, 139.7454),  # Tokyo Tower
            (35.6605, 139.7292),  # Roppongi
            (35.6594, 139.7370),  # Azabudai
            (35.6670, 139.7495),  # Toranomon
            (35.6750, 139.7550),  # C1 Loop
            (35.6812, 139.7671),  # Tokyo Station
            (35.6984, 139.7731),  # Akihabara
            (35.7101, 139.8107),  # Skytree
        ],
    },
    {
        'name': 'Pedestrian (Ginza→Tokyo Station)',
        'type': 'pedestrian',
        'speed_kmh': 5,
        'waypoints': [
            (35.6717, 139.7649),  # Ginza
            (35.6730, 139.7660),
            (35.6750, 139.7665),
            (35.6770, 139.7668),
            (35.6790, 139.7670),
            (35.6812, 139.7671),  # Tokyo Station
        ],
    },
    {
        'name': 'Roppongi→Shinagawa (mixed urban)',
        'type': 'vehicle',
        'speed_kmh': 30,
        'waypoints': [
            (35.6605, 139.7292),  # Roppongi
            (35.6594, 139.7370),  # Azabudai
            (35.6560, 139.7350),
            (35.6500, 139.7380),
            (35.6420, 139.7390),
            (35.6350, 139.7387),
            (35.6284, 139.7387),  # Shinagawa
        ],
    },
    {
        'name': 'Odaiba→Central (waterfront)',
        'type': 'vehicle',
        'speed_kmh': 50,
        'waypoints': [
            (35.6190, 139.7760),  # Odaiba
            (35.6280, 139.7700),
            (35.6400, 139.7600),
            (35.6500, 139.7550),
            (35.6586, 139.7454),  # Tokyo Tower
            (35.6670, 139.7495),  # Toranomon
        ],
    },
]

def interpolate_route(waypoints, interval_m=50):
    """Interpolate waypoints at fixed meter intervals using UTM."""
    points = []
    for i in range(len(waypoints) - 1):
        lat1, lng1 = waypoints[i]
        lat2, lng2 = waypoints[i + 1]
        x1, y1 = to_local(lat1, lng1)
        x2, y2 = to_local(lat2, lng2)
        dist = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
        n_steps = max(1, int(dist / interval_m))
        for s in range(n_steps):
            frac = s / n_steps
            x = x1 + frac * (x2 - x1)
            y = y1 + frac * (y2 - y1)
            lat, lng = to_latlng(x, y)
            points.append((lat, lng, x, y))
    lat_end, lng_end = waypoints[-1]
    x_end, y_end = to_local(lat_end, lng_end)
    points.append((lat_end, lng_end, x_end, y_end))
    return points

def sample_rsrp_at_point(x, y, rss_dbm_grid, cols, rows):
    """Sample RSRP from coverage map at a local (x, y) coordinate."""
    x_min = -((cols * 10) / 2)
    y_min = -((rows * 10) / 2)
    c = int((x - x_min) / 10)
    r = int((y - y_min) / 10)
    if 0 <= r < rows and 0 <= c < cols:
        return float(rss_dbm_grid[r, c])
    return -140.0  # out of coverage

def get_serving_cell(x, y, rss_per_tx, cols, rows):
    """Return index of strongest TX at (x, y)."""
    x_min = -((cols * 10) / 2)
    y_min = -((rows * 10) / 2)
    c = int((x - x_min) / 10)
    r = int((y - y_min) / 10)
    if 0 <= r < rows and 0 <= c < cols:
        cell_rss = rss_per_tx[:, r, c]
        return int(np.argmax(cell_rss))
    return -1

# Prepare per-TX RSS grid for serving cell detection
rss_all = rm_n77.rss.numpy()
rss_per_tx = 10 * np.log10(np.clip(rss_all, 1e-30, None)) + 30
rss_best_dbm = rss_per_tx.max(axis=0).squeeze()
n_rows, n_cols = rss_best_dbm.shape

# Build TX index → tower/sector mapping
tx_names = []
for t in towers:
    for s_idx in range(len(t['azimuth'])):
        tx_names.append(f"{t['id']}-S{s_idx}")

# --- Simulate all routes ---
all_traces = []
total_ho = {'success': 0, 'failure': 0, 'pingpong': 0}

for route in routes:
    points = interpolate_route(route['waypoints'], interval_m=50)
    trace = []
    prev_cell = -1
    ho_history = []  # last N serving cells for ping-pong detection

    for idx, (lat, lng, x, y) in enumerate(points):
        rsrp = sample_rsrp_at_point(x, y, rss_best_dbm, n_cols, n_rows)
        cell = get_serving_cell(x, y, rss_per_tx.squeeze() if rss_per_tx.ndim > 3 else rss_per_tx, n_cols, n_rows)

        ho_event = None
        if prev_cell >= 0 and cell != prev_cell:
            if rsrp < -120:
                ho_event = 'failure'
                total_ho['failure'] += 1
            elif len(ho_history) >= 3 and cell in ho_history[-3:]:
                ho_event = 'pingpong'
                total_ho['pingpong'] += 1
            else:
                ho_event = 'success'
                total_ho['success'] += 1

        ho_history.append(cell)
        if len(ho_history) > 10:
            ho_history.pop(0)

        cell_name = tx_names[cell] if 0 <= cell < len(tx_names) else 'none'
        trace.append({
            'lat': lat, 'lng': lng, 'rsrp': rsrp,
            'cell': cell_name, 'ho_event': ho_event,
            'distance_m': idx * 50,
        })
        prev_cell = cell

    all_traces.append({'route': route, 'trace': trace})
    n_ho = sum(1 for p in trace if p['ho_event'])
    print(f"Route: {route['name']}")
    print(f"  Points: {len(trace)} | Handovers: {n_ho} | Avg RSRP: {np.mean([p['rsrp'] for p in trace]):.1f} dBm")

print(f'\n=== Aggregate HO Statistics ===')
print(f"  Success:  {total_ho['success']}")
print(f"  Failure:  {total_ho['failure']} (RSRP < -120 dBm at HO)")
print(f"  Pingpong: {total_ho['pingpong']} (returned to prev cell within 3 steps)")
ho_total = sum(total_ho.values())
if ho_total > 0:
    print(f"  Success rate: {total_ho['success']/ho_total*100:.1f}%")
    print(f"  Failure rate: {total_ho['failure']/ho_total*100:.1f}%")

# --- Export as GeoJSON for Cesium ---
features = []
for trace_data in all_traces:
    route = trace_data['route']
    trace = trace_data['trace']

    # Route line
    coords = [[p['lng'], p['lat']] for p in trace]
    features.append(gj.Feature(
        geometry=gj.LineString(coords),
        properties={
            'name': route['name'],
            'type': route['type'],
            'speed_kmh': route['speed_kmh'],
            'n_handovers': sum(1 for p in trace if p['ho_event']),
            'stroke': '#00b4d8' if route['type'] == 'train' else '#ffca28' if route['type'] == 'vehicle' else '#e040fb',
            'stroke-width': 3,
        }
    ))

    # HO event markers
    for p in trace:
        if p['ho_event']:
            color = '#00e676' if p['ho_event'] == 'success' else '#ff5252' if p['ho_event'] == 'failure' else '#ff9800'
            features.append(gj.Feature(
                geometry=gj.Point([p['lng'], p['lat']]),
                properties={
                    'ho_event': p['ho_event'],
                    'rsrp': round(p['rsrp'], 1),
                    'cell': p['cell'],
                    'route': route['name'],
                    'marker-color': color,
                    'marker-size': 'medium' if p['ho_event'] == 'failure' else 'small',
                }
            ))

ue_geojson_path = 'tokyo_ue_mobility.geojson'
with open(ue_geojson_path, 'w') as f:
    gj.dump(gj.FeatureCollection(features), f)

print(f'\nExported UE mobility traces to {ue_geojson_path}')
print(f'  {len(routes)} routes, {sum(len(t["trace"]) for t in all_traces)} waypoints')
print(f'  Cesium: load as GeoJSON data source alongside coverage layers')

# --- RSRP along route plot ---
fig, axes = plt.subplots(len(routes), 1, figsize=(14, 3 * len(routes)), sharex=False)
if len(routes) == 1:
    axes = [axes]

for ax, trace_data in zip(axes, all_traces):
    route = trace_data['route']
    trace = trace_data['trace']
    distances = [p['distance_m'] for p in trace]
    rsrps = [p['rsrp'] for p in trace]

    ax.plot(distances, rsrps, linewidth=1.5, color='#00b4d8')
    ax.axhline(y=-120, color='red', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.axhline(y=-100, color='orange', linestyle='--', alpha=0.5, linewidth=0.8)

    for p in trace:
        if p['ho_event'] == 'failure':
            ax.axvline(x=p['distance_m'], color='red', alpha=0.6, linewidth=1)
        elif p['ho_event'] == 'pingpong':
            ax.axvline(x=p['distance_m'], color='orange', alpha=0.4, linewidth=1)
        elif p['ho_event'] == 'success':
            ax.axvline(x=p['distance_m'], color='green', alpha=0.2, linewidth=0.5)

    ax.set_ylabel('RSRP (dBm)')
    ax.set_title(f"{route['name']} ({route['type']}, {route['speed_kmh']} km/h)", fontsize=10)
    ax.set_ylim(-140, -40)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Distance (m)')
plt.suptitle('WINNIIO Tokyo MRO — UE RSRP Along Route (HO events marked)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('tokyo_ue_rsrp_traces.png', dpi=150, bbox_inches='tight')
plt.show()

try:
    files.download(ue_geojson_path)
    files.download('tokyo_ue_rsrp_traces.png')
except:
    pass

## 15b. ROI Calculator — HO Failure Reduction → OpEx Savings

Translates simulation results into business impact. Uses Rakuten Mobile's published financials (FY2025: JPY 375B revenue, 10M+ subscribers, EBITDA JPY 12.9B) and industry benchmarks for HO failure cost.

In [ ]:
# --- ROI Calculator: MRO Optimization → Business Impact ---
# Sources: Rakuten Mobile FY2025 financials, McKinsey telecom OpEx benchmarks,
# 3GPP TS 36.331 / 38.331 HO failure definitions

# --- Network parameters (Rakuten Mobile Japan, public data) ---
total_sites = 50_000          # Rakuten Mobile cell sites (estimated from 4G+5G rollout)
subs = 10_000_000             # 10M+ subscribers (FY2025)
revenue_jpy = 375_000_000_000 # JPY 375B annual
opex_jpy = 362_000_000_000    # JPY 362B (revenue - EBITDA 12.9B)
jpy_eur = 0.0062              # approximate

# --- HO failure economics (industry benchmarks) ---
# Each HO failure triggers: RRC re-establishment (~2s), possible RLF, ticket/NOC cost
# McKinsey: 15-25% of mobile OpEx is RAN optimization related
ran_opex_pct = 0.20
ran_opex = opex_jpy * ran_opex_pct

# HO failure rate benchmarks (3GPP, real network data)
baseline_ho_failure_pct = 2.5   # typical: 1-5% depending on density/mobility
target_ho_failure_pct = 0.5     # achievable with ML-optimized CIO: <1%

# Cost per HO failure event
ho_per_sub_per_day = 8          # avg handovers per subscriber per day (urban dense)
ho_events_daily = subs * ho_per_sub_per_day
ho_failures_daily_baseline = ho_events_daily * (baseline_ho_failure_pct / 100)
ho_failures_daily_target = ho_events_daily * (target_ho_failure_pct / 100)
failures_avoided_daily = ho_failures_daily_baseline - ho_failures_daily_target

# Each failure = ~30 sec degraded experience + NOC triage cost
# Conservative: JPY 0.5 per failure (blend of churn risk + NOC + QoE degradation)
cost_per_failure_jpy = 0.5
savings_daily_jpy = failures_avoided_daily * cost_per_failure_jpy
savings_annual_jpy = savings_daily_jpy * 365
savings_annual_eur = savings_annual_jpy * jpy_eur

# Churn reduction value
# 0.1% churn reduction from better HO = massive LTV impact
arpu_monthly_jpy = revenue_jpy / subs / 12
churn_reduction_pct = 0.1
churn_saved_subs = subs * (churn_reduction_pct / 100)
churn_ltv_jpy = churn_saved_subs * arpu_monthly_jpy * 24  # 24-month LTV
churn_ltv_eur = churn_ltv_jpy * jpy_eur

print('=' * 65)
print('  WINNIIO MRO Digital Twin — ROI Calculator')
print('=' * 65)
print(f'\n--- Network Scale ---')
print(f'  Sites: {total_sites:,} | Subscribers: {subs/1e6:.0f}M')
print(f'  Revenue: JPY {revenue_jpy/1e9:.0f}B (EUR {revenue_jpy*jpy_eur/1e6:.0f}M)')
print(f'  RAN OpEx ({ran_opex_pct*100:.0f}% of total): JPY {ran_opex/1e9:.0f}B')

print(f'\n--- Handover Economics ---')
print(f'  Daily HO events: {ho_events_daily/1e6:.0f}M')
print(f'  Baseline failure rate: {baseline_ho_failure_pct}% → {ho_failures_daily_baseline/1e6:.1f}M failures/day')
print(f'  Target failure rate:   {target_ho_failure_pct}% → {ho_failures_daily_target/1e6:.1f}M failures/day')
print(f'  Failures avoided/day:  {failures_avoided_daily/1e6:.1f}M')

print(f'\n--- Annual Savings ---')
print(f'  Direct OpEx savings:     JPY {savings_annual_jpy/1e9:.1f}B (EUR {savings_annual_eur/1e6:.1f}M)')
print(f'  Churn reduction ({churn_reduction_pct}%):  JPY {churn_ltv_jpy/1e9:.1f}B LTV (EUR {churn_ltv_eur/1e6:.1f}M)')
print(f'  Combined impact:         EUR {(savings_annual_eur + churn_ltv_eur)/1e6:.1f}M / year')

print(f'\n--- Phase 1+2 Investment ---')
print(f'  WINNIIO engagement:      EUR 25,000')
print(f'  ROI multiple:            {(savings_annual_eur + churn_ltv_eur) / 25000:.0f}x')
print(f'  Payback:                 < 1 day of optimized operation')

print(f'\n--- Per-Site Economics ---')
print(f'  Annual savings per site: EUR {(savings_annual_eur + churn_ltv_eur) / total_sites:.0f}')
print(f'  WINNIIO license target:  ~$10/site/month = $6M ARR at scale')

print(f'\n⚠️  Estimates use public benchmarks. Phase 1 calibrates with real KPIs.')
print(f'    Replace baseline_ho_failure_pct with actual Rakuten MRO data.')

## 17. Competitive Positioning + 90-Day Success Criteria

### Why WINNIIO vs. alternatives

| Approach | Strengths | Limitations | WINNIIO Advantage |
|----------|-----------|-------------|-------------------|
| **Atoll / ASSET** (Forsk/Aircom) | Industry standard, mature, operator trust | Empirical propagation (Okumura-Hata), no ray-tracing, no DT, expensive license | Physics-based RT, open-source, 3D city twin |
| **AWE Communications** (Altair) | Good ray-tracing engine | Proprietary, no RL/ML integration, no CityGML pipeline | End-to-end: RT → HO analysis → RL → xApp |
| **Planet (Infovista)** | Large install base, field-proven | Legacy architecture, slow innovation, vendor lock-in | Open stack, methodology-first, runs on Intel COTS |
| **In-house (RCP team)** | Full control, integrated with Altiostar stack | Requires 6-12 months + headcount, diverts from core RAN | EUR 25K gets working prototype in 30 days |
| **NVIDIA Aerial/Omniverse** | GPU-native, photorealistic, Sionna integration | Requires Omniverse license, heavy compute, NVIDIA dependency | We USE Sionna (Apache 2.0) without Omniverse lock-in |

### 90-Day Success Criteria (Phase 1 + Phase 2)

**Day 0-15 (Phase 1 — Reality Emulation Workshop):**
- [ ] Altiostar provides: site database, antenna files, MDT/CHR data for Tokyo cluster
- [ ] SPIN Twinning workshop: walk through THIS notebook with engineering team
- [ ] Agree on: KPIs, target HO failure rate, calibration acceptance criteria
- [ ] Deliverable: signed-off gap analysis + Phase 2 scope

**Day 15-60 (Phase 2 — Concurrent Engineering Sprint):**
- [ ] PLATEAU LOD2 full ingestion for target area
- [ ] Calibration RMSE < 8 dB against real MDT data
- [ ] Real antenna patterns loaded (vendor .msi/.ffe files)
- [ ] RL agent trained on simulated environment (Stable Baselines3)
- [ ] Shadow-mode xApp prototype on near-RT RIC testbed
- [ ] Deliverable: calibrated digital twin + trained MRO model + deployment plan

**Day 60-90 (Phase 2 wrap + Phase 3 scoping):**
- [ ] A/B comparison: simulated vs. real HO stats (correlation > 0.8)
- [ ] Phase 3 scope defined by Phase 1+2 output (not pre-defined)
- [ ] Business case validated with real numbers (replace ROI calculator estimates)
- [ ] Decision: advisory engagement vs. WINNIIO-led platform build

### What EUR 25,000 buys
1. **Calibrated 3D digital twin** of your Tokyo network with ray-traced RF
2. **HO analysis** with real UE traces on real building geometry
3. **Trained RL agent** for CIO optimization (simulation environment)
4. **Shared reality agreement** — your team and ours aligned on what matters
5. **Phase 3 scope** — defined by evidence, not assumptions

## 18. What This Proves — Cesium Integration

All GeoJSON files serve alongside `altiostar-tokyo-demo.html` — toggle button + band selector already integrated.

### Demonstrated capabilities:

1. **Shared reality canvas** — Altiostar's Tokyo network in 3D with ray-traced RF
2. **Physics-based RF** — ITU materials (concrete + glass), diffraction, 10^6 rays
3. **Multi-band** — n77, n78, n257 with frequency-dependent materials
4. **Custom antennas** — 3GPP TR 38.901, 65° HPBW, 18 dBi, 30 dB F/B
5. **HO boundary visibility** — TX association map shows handover zones
6. **UE mobility traces** — 5 routes, RSRP sampling, HO event classification
7. **ROI calculator** — HO failure % → OpEx savings with Rakuten financials
8. **Calibration path** — MDT import, RMSE/MAE/bias metrics
9. **RL hook** — MRO reward function, Stable Baselines3 ready
10. **Open stack** — Sionna + PLATEAU + CesiumJS, all Apache 2.0
11. **Competitive positioning** — vs Atoll, AWE, Planet, in-house, NVIDIA
12. **90-day roadmap** — clear success criteria per phase

**Next step:** Pre-workshop call with Altiostar engineering team → walk through this notebook → SPIN Twinning.